# Verificación reproducible del Taller 3

**Autores:** Jairo Serrano y Edwin Yunis.

Cuaderno complementario del dashboard y Jupyter Book. Utiliza únicamente la matriz del Excel original exportada a CSV. No requiere bibliotecas externas. Ejecútalo en Jupyter desde la carpeta `book/` del repositorio.

## 1. Importación y validación de los datos

Se comprueba la dimensión 12×12 y que la matriz contiene 43 intereses.

In [ ]:
import csv
from pathlib import Path
from itertools import combinations

ruta = Path('../data/matriz_intereses.csv')
if not ruta.exists():
    ruta = Path('data/matriz_intereses.csv')
with ruta.open(encoding='utf-8-sig', newline='') as f:
    filas = list(csv.reader(f))
tematicas = filas[0][1:]
sectores = [r[0] for r in filas[1:]]
A = [[int(v) for v in r[1:]] for r in filas[1:]]
assert len(A) == 12 and all(len(r) == 12 for r in A)
assert sum(map(sum, A)) == 43
print('Sectores:', len(sectores), 'Temáticas:', len(tematicas), 'Intereses:', sum(map(sum, A)))

## 2. Algoritmo de ordenamiento binario (ROC)

Pesos descendentes de potencias de dos y alternancia de ordenamientos de filas y columnas hasta lograr estabilidad.

In [ ]:
def reorder_roc(matriz):
    m = [row[:] for row in matriz]
    row_ids = list(range(len(m)))
    col_ids = list(range(len(m[0])))
    history = []
    for step in range(100):
        row_scores = [sum(v * 2**(len(m[0])-j-1) for j,v in enumerate(row)) for row in m]
        row_order = sorted(range(len(m)),key=lambda i:-row_scores[i])
        m = [m[i] for i in row_order]
        new_row_ids = [row_ids[i] for i in row_order]
        col_scores = [sum(m[i][j]*2**(len(m)-i-1) for i in range(len(m))) for j in range(len(m[0]))]
        col_order = sorted(range(len(m[0])),key=lambda j:-col_scores[j])
        m = [[row[j] for j in col_order] for row in m]
        new_col_ids = [col_ids[j] for j in col_order]
        history.append((new_row_ids[:],new_col_ids[:]))
        if new_row_ids == row_ids and new_col_ids == col_ids: break
        row_ids,col_ids=new_row_ids,new_col_ids
    return m,row_ids,col_ids,history

roc_matrix, row_order, col_order, history = reorder_roc(A)
print('Iteraciones:', len(history))
print('Orden final de temáticas:')
for j in col_order: print('-', tematicas[j])

## 3. Evaluación ROC y prueba exhaustiva de optimización

Se enumeran las 15.400 particiones de doce temáticas en cuatro grupos no etiquetados de tres. La cota inferior por sector verifica el óptimo.

In [ ]:
def loss_for_groups(groups):
    return sum(sum(max(0,sum(row[j] for j in group)-1) for group in groups) for row in A)

roc_groups = [col_order[k:k+3] for k in range(0,12,3)]
print('Pérdidas ROC:', loss_for_groups(roc_groups))

# Enumeración sin repetir permutaciones de franjas ni auditorios.
counter = 0
best = float('inf')
best_groups = []
def search(remaining, groups):
    global counter, best, best_groups
    if not remaining:
        counter += 1
        loss = loss_for_groups(groups)
        if loss < best:
            best = loss
            best_groups = [tuple(tuple(x) for x in groups)]
        elif loss == best:
            best_groups.append(tuple(tuple(x) for x in groups))
        return
    first = min(remaining)
    for pair in combinations([v for v in remaining if v!=first],2):
        group = (first,)+pair
        search([v for v in remaining if v not in group], groups+[group])

search(list(range(12)),[])
print('Particiones estudiadas:',counter)
print('Pérdidas óptimas:',best)
print('Número de particiones óptimas:',len(best_groups))
assert counter == 15400 and best == 7
lower_bound = sum(max(0,sum(row)-4) for row in A)
print('Cota inferior teórica:',lower_bound)
assert lower_bound == best

## 4. Programación recomendada

Un representante sectorial puede asistir a una conferencia por franja; 7 pérdidas es el mínimo global.

In [ ]:
optimal = [[2,9,10],[0,3,7],[5,8,11],[1,4,6]]
hours = ['08:00–09:45','10:00–11:45','14:00–15:45','16:00–17:45']
assert loss_for_groups(optimal) == 7
print('Programa optimizado:')
for hour,group in zip(hours,optimal):
    print(hour, ' | '.join(tematicas[j] for j in group))
print('Aprovechamiento:',43-loss_for_groups(optimal),'de 43')

## Conclusión

El procedimiento ROC ofrece una organización por afinidad, pero la optimización reduce las oportunidades perdidas de 18 a 7, es decir, **61,11 %**.

El análisis es reproducible a partir de los datos del taller académico.